# T12E CNN-2D Interpretable Features Stepwise SVR

CNN-2Dの解釈結果から選んだ water / phase / structure / scatter / interaction / CNN woodtype probability 特徴量を作成し、GroupKFold by 樹種でSVRをstep-wise評価します。

重い場合は `QUICK_MODE=True` のまま実行します。Colabで広く探索する場合は `QUICK_MODE=False`、`MAX_STEPS`、`TOP_K_PREFILTER` を調整してください。

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [
        start,
        *start.parents,
        Path("/content/NIR_CNN_2D_T12E_colab"),
        Path("/content/NIR_CNN_2D"),
        Path("/content/drive/MyDrive/NIR_CNN_2D_T12E_colab"),
        Path("/content/drive/MyDrive/NIR_CNN_2D"),
    ]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate.resolve()
    raise FileNotFoundError("Project root not found. Run this notebook inside the repository.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import DataConfig
from src.data.preprocessing import savitzky_golay_derivative, snv
from src.utils.plotting import configure_matplotlib_japanese

sns.set_theme(style="whitegrid")
configure_matplotlib_japanese()

config = DataConfig()
ID_COL = "sample number"
GROUP_COL = "species number"
SPECIES_COL = "樹種"
TARGET_COL = "含水率"

def running_on_colab() -> bool:
    return Path("/content").exists() and Path("/content/drive/MyDrive").exists()


def make_result_dir() -> Path:
    if running_on_colab():
        # Keep outputs in a Drive folder independent from the input bundle.
        out = Path("/content/drive/MyDrive/NIR_CNN_2D_T12E_results") / "T12E_cnn2d_interpretable_stepwise"
    else:
        out = PROJECT_ROOT / "outputs" / "T12E_cnn2d_interpretable_stepwise"
    out.mkdir(parents=True, exist_ok=True)
    return out


RESULT_DIR = make_result_dir()
FIGURE_DIR = RESULT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

QUICK_MODE = True
TOP_K_PREFILTER = 28 if QUICK_MODE else None
MAX_STEPS = 12 if QUICK_MODE else 25
MIN_IMPROVEMENT = 0.0
N_SPLITS = 5

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RESULT_DIR =", RESULT_DIR)
print("QUICK_MODE =", QUICK_MODE, "TOP_K_PREFILTER =", TOP_K_PREFILTER, "MAX_STEPS =", MAX_STEPS)

## Utilities

In [ ]:
SOFTWOOD_CANDIDATES = ["softwood_prob", "prob_softwood", "p_softwood", "woodtype_softwood_prob", "prob_woodtype_softwood"]
HARDWOOD_CANDIDATES = ["hardwood_prob", "prob_hardwood", "p_hardwood", "woodtype_hardwood_prob", "prob_woodtype_hardwood"]


def savefig(name: str) -> Path:
    path = FIGURE_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=130, bbox_inches="tight")
    print("saved:", path)
    return path


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2)))


def read_header(path: Path) -> pd.Index:
    return pd.read_csv(path, encoding="cp932", nrows=0, engine="python").columns


def spectral_cols(header: pd.Index) -> list[str]:
    out = []
    for col in header:
        try:
            float(col)
            out.append(col)
        except ValueError:
            pass
    return out


def read_meta(path: Path, include_target: bool) -> pd.DataFrame:
    header = read_header(path)
    cols = [c for c in [ID_COL, GROUP_COL, SPECIES_COL, TARGET_COL] if c in header and (include_target or c != TARGET_COL)]
    return pd.read_csv(path, encoding="cp932", usecols=cols)


def read_spectra(path: Path) -> tuple[np.ndarray, np.ndarray]:
    header = read_header(path)
    cols = spectral_cols(header)
    df = pd.read_csv(path, encoding="cp932", usecols=cols, dtype={c: "float32" for c in cols}, engine="python")
    wavenumbers = np.asarray([float(c) for c in cols], dtype=float)
    wavelengths = 1e7 / wavenumbers
    order = np.argsort(wavelengths)
    return df[[cols[i] for i in order]].to_numpy(dtype=np.float32), wavelengths[order]


def band_mask(wl, low, high):
    return (wl >= low) & (wl <= high)


def band_mean(X, wl, low, high):
    m = band_mask(wl, low, high)
    return np.nanmean(X[:, m], axis=1) if m.any() else np.full(X.shape[0], np.nan)


def band_area(X, wl, low, high):
    m = band_mask(wl, low, high)
    return np.trapezoid(X[:, m], wl[m], axis=1) if m.sum() >= 2 else np.full(X.shape[0], np.nan)


def center_mean(X, wl, center, width=10):
    return band_mean(X, wl, center - width, center + width)


def nearest_val(X, wl, target):
    return X[:, int(np.argmin(np.abs(wl - target)))]


def band_slope(X, wl, low, high):
    return (center_mean(X, wl, high, 5) - center_mean(X, wl, low, 5)) / max(high - low, 1e-8)


def point_diff(X, wl, left, right, width=10):
    return center_mean(X, wl, right, width) - center_mean(X, wl, left, width)


def safe_ratio(a, b, eps=1e-8):
    return np.asarray(a, dtype=float) / (np.asarray(b, dtype=float) + eps)


def read_csv_auto(path: Path) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp932"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path)


def pick_col(df, candidates, keyword):
    for col in candidates:
        if col in df.columns:
            return col
    for col in df.columns:
        if keyword.lower() in str(col).lower():
            return col
    return None


def find_probability_frame(split: str) -> pd.DataFrame:
    explicit = {
        "train": PROJECT_ROOT / "outputs" / "T12_cnn_soft_routing" / "oof" / "train_woodtype_oof_probs.csv",
        "test": PROJECT_ROOT / "outputs" / "T12_cnn_soft_routing" / "test" / "test_woodtype_probs.csv",
    }[split]
    candidates = [explicit, *list((PROJECT_ROOT / "outputs").glob(f"**/*{split}*woodtype*prob*.csv"))]
    seen = set()
    for path in candidates:
        if path in seen or not path.exists():
            continue
        seen.add(path)
        df = read_csv_auto(path)
        if pick_col(df, SOFTWOOD_CANDIDATES, "softwood") or pick_col(df, HARDWOOD_CANDIDATES, "hardwood"):
            print(f"loaded {split} probability:", path)
            return df
    raise FileNotFoundError(f"{split} woodtype probability file was not found")


def attach_probability_features(feat: pd.DataFrame, prob_df: pd.DataFrame) -> pd.DataFrame:
    out = feat.copy()
    prob = prob_df.copy()
    soft_col = pick_col(prob, SOFTWOOD_CANDIDATES, "softwood")
    hard_col = pick_col(prob, HARDWOOD_CANDIDATES, "hardwood")
    if ID_COL in out.columns and ID_COL in prob.columns:
        keep = [ID_COL] + [c for c in [soft_col, hard_col] if c is not None]
        out = out.merge(prob[keep], on=ID_COL, how="left")
    else:
        if len(prob) != len(out):
            raise ValueError("Probability frame length does not match and sample number is unavailable")
        if soft_col:
            out[soft_col] = prob[soft_col].values
        if hard_col:
            out[hard_col] = prob[hard_col].values
    soft_col = pick_col(out, SOFTWOOD_CANDIDATES, "softwood")
    hard_col = pick_col(out, HARDWOOD_CANDIDATES, "hardwood")
    out["softwood_prob"] = pd.to_numeric(out[soft_col], errors="coerce") if soft_col else 1.0 - pd.to_numeric(out[hard_col], errors="coerce")
    out["hardwood_prob"] = pd.to_numeric(out[hard_col], errors="coerce") if hard_col else 1.0 - out["softwood_prob"]
    total = (out["softwood_prob"] + out["hardwood_prob"]).replace(0, np.nan)
    out["softwood_prob"] = (out["softwood_prob"] / total).clip(0, 1)
    out["hardwood_prob"] = (out["hardwood_prob"] / total).clip(0, 1)
    probs = out[["softwood_prob", "hardwood_prob"]].clip(1e-8, 1.0).to_numpy(float)
    out["entropy_woodtype"] = -np.sum(probs * np.log(probs), axis=1)
    out["uncertainty_woodtype"] = 1.0 - np.nanmax(probs, axis=1)
    return out

## Build CNN-2D Interpretable Feature Table

In [ ]:
def add_feature(out, rows, feature, values, system, preprocess, wavelength, meaning):
    out[feature] = values
    rows.append({"feature": feature, "system": system, "preprocess": preprocess, "wavelength": wavelength, "meaning": meaning})


def pf(preprocess, base):
    return f"{preprocess}_{base}"


def build_feature_table(base_df, X_raw, wl, X_snv, X_sg1, X_sg2, X_snv_sg1, X_snv_sg2):
    out = pd.DataFrame()
    out[ID_COL] = base_df[ID_COL].values
    if GROUP_COL in base_df.columns:
        out[GROUP_COL] = base_df[GROUP_COL].values
    out[SPECIES_COL] = base_df[SPECIES_COL].values
    if TARGET_COL in base_df.columns:
        out[TARGET_COL] = base_df[TARGET_COL].values
    rows = []
    views = {"raw": X_raw, "snv": X_snv, "raw_sg1": X_sg1, "raw_sg2": X_sg2, "snv_sg1": X_snv_sg1, "snv_sg2": X_snv_sg2}

    for prep in ["raw", "snv", "raw_sg1", "raw_sg2"]:
        add_feature(out, rows, pf(prep, "mean_1400_1500"), band_mean(views[prep], wl, 1400, 1500), "water_1450", prep, "1400-1500", "1450nm水吸収")
        add_feature(out, rows, pf(prep, "mean_1850_1950"), band_mean(views[prep], wl, 1850, 1950), "water_1900", prep, "1850-1950", "1900nm水吸収")
        add_feature(out, rows, pf(prep, "mean_2200_2300"), band_mean(views[prep], wl, 2200, 2300), "structure", prep, "2200-2300", "木材構造")
        add_feature(out, rows, pf(prep, "mean_2300_2400"), band_mean(views[prep], wl, 2300, 2400), "structure", prep, "2300-2400", "樹種/構造差")
        add_feature(out, rows, pf(prep, "mean_2400_2500"), band_mean(views[prep], wl, 2400, 2500), "structure", prep, "2400-2500", "高波長構造差")

    for prep in ["raw", "snv"]:
        X = views[prep]
        p1450 = center_mean(X, wl, 1450, 10)
        p1900 = center_mean(X, wl, 1900, 10)
        m1400 = band_mean(X, wl, 1400, 1500)
        m1900 = band_mean(X, wl, 1900, 2000)
        add_feature(out, rows, pf(prep, "peak_1450"), p1450, "water_1450", prep, "1450", "水ピーク強度")
        add_feature(out, rows, pf(prep, "band_area_1400_1500"), band_area(X, wl, 1400, 1500), "water_1450", prep, "1400-1500", "水吸収面積")
        add_feature(out, rows, pf(prep, "peak_1900"), p1900, "water_1900", prep, "1900", "強水吸収")
        add_feature(out, rows, pf(prep, "band_area_1850_1950"), band_area(X, wl, 1850, 1950), "water_1900", prep, "1850-1950", "水吸収面積")
        add_feature(out, rows, pf(prep, "ratio_1450_1900"), safe_ratio(p1450, p1900), "water_ratio", prep, "1450/1900", "結合水/自由水バランス")
        add_feature(out, rows, pf(prep, "meanDiff_1400_1500__1900_2000"), m1900 - m1400, "water_diff", prep, "1400-1500 vs 1900-2000", "既存best水特徴")
        add_feature(out, rows, pf(prep, "mean_2100_2300"), band_mean(X, wl, 2100, 2300), "structure", prep, "2100-2300", "lignin/cellulose構造")
        add_feature(out, rows, pf(prep, "mean_1000_1300"), band_mean(X, wl, 1000, 1300), "scatter", prep, "1000-1300", "光輸送")
        add_feature(out, rows, pf(prep, "d_1200_2100"), center_mean(X, wl, 1200, 10) - center_mean(X, wl, 2100, 10), "scatter", prep, "1200 vs 2100", "既存best候補")
        add_feature(out, rows, pf(prep, "pointDiff_1200_2200"), point_diff(X, wl, 1200, 2200, 10), "scatter", prep, "1200 vs 2200", "既存best候補")
        add_feature(out, rows, pf(prep, "B1900_1970"), band_mean(X, wl, 1900, 1970), "interaction", prep, "1900-1970", "水吸収形状")

    add_feature(out, rows, "raw_sg1_slope_1400_1500", band_slope(X_sg1, wl, 1400, 1500), "water_1450", "raw_sg1", "1400-1500", "水吸収の立ち上がり")
    add_feature(out, rows, "raw_sg1_slope_1850_1950", band_slope(X_sg1, wl, 1850, 1950), "water_1900", "raw_sg1", "1850-1950", "水吸収勾配")
    add_feature(out, rows, "raw_sg2_curv_1450", center_mean(X_sg2, wl, 1450, 10), "water_1450", "raw_sg2", "1450", "水ピーク曲率")
    add_feature(out, rows, "raw_sg2_curv_1900", center_mean(X_sg2, wl, 1900, 10), "water_1900", "raw_sg2", "1900", "水ピーク曲率")

    add_feature(out, rows, "sg1_2100", nearest_val(X_sg1, wl, 2100), "phase", "raw_sg1", "2100", "乾燥phase")
    add_feature(out, rows, "raw_sg1_sg1_mean_2010_2035", band_mean(X_sg1, wl, 2010, 2035), "phase", "raw_sg1", "2010-2035", "phase変化")
    add_feature(out, rows, "raw_sg1_sg1_mean_2090_2120", band_mean(X_sg1, wl, 2090, 2120), "phase", "raw_sg1", "2090-2120", "phase変化")
    add_feature(out, rows, "raw_sg1_sg1_slope_2090_2120", band_slope(X_sg1, wl, 2090, 2120), "phase", "raw_sg1", "2090-2120", "FSP遷移")
    add_feature(out, rows, "raw_sg1_sg1_diff_2030_2100", point_diff(X_sg1, wl, 2030, 2100, 5), "phase", "raw_sg1", "2030 vs 2100", "乾燥状態差")
    add_feature(out, rows, "raw_sg2_sg2_curv_2030_2100", band_mean(X_sg2, wl, 2030, 2100), "phase", "raw_sg2", "2030-2100", "phase曲率")

    add_feature(out, rows, "raw_mean_1000_1100", band_mean(X_raw, wl, 1000, 1100), "scatter", "raw", "1000-1100", "散乱/透過")
    for prep, X in [("raw_sg1", X_sg1), ("raw_sg2", X_sg2)]:
        add_feature(out, rows, pf(prep, "slope_1000_1300"), band_slope(X, wl, 1000, 1300), "scatter", prep, "1000-1300", "optical clearing")

    for target in [2246, 2250, 2300, 2380, 2420]:
        add_feature(out, rows, f"raw_sg2_sg2_{target}", nearest_val(X_sg2, wl, target), "structure", "raw_sg2", str(target), "CNN-2D解釈で強い構造点")

    snv1450 = center_mean(X_snv, wl, 1450, 10)
    snv1900 = center_mean(X_snv, wl, 1900, 10)
    sg1_2100 = out["sg1_2100"].to_numpy(float)
    sg2_2300 = nearest_val(X_sg2, wl, 2300)
    add_feature(out, rows, "snv1450_x_sg1_2100", snv1450 * sg1_2100, "interaction", "snv_x_raw_sg1", "1450 x 2100", "water x phase")
    add_feature(out, rows, "snv1900_x_sg1_2100", snv1900 * sg1_2100, "interaction", "snv_x_raw_sg1", "1900 x 2100", "water x phase")
    add_feature(out, rows, "snv1900_x_sg2_2300", snv1900 * sg2_2300, "interaction", "snv_x_raw_sg2", "1900 x 2300", "water x structure")
    return out, pd.DataFrame(rows)


train_path = PROJECT_ROOT / "data" / config.train_csv
test_path = PROJECT_ROOT / "data" / config.test_csv
train_meta = read_meta(train_path, include_target=True)
test_meta = read_meta(test_path, include_target=False)
X_raw_train, wavelengths = read_spectra(train_path)
X_raw_test, wavelengths_test = read_spectra(test_path)
assert np.allclose(wavelengths, wavelengths_test)

X_snv_train = snv(X_raw_train)
X_snv_test = snv(X_raw_test)
X_sg1_train = savitzky_golay_derivative(X_raw_train, 1, config.sg_window_length, config.sg_polyorder)
X_sg1_test = savitzky_golay_derivative(X_raw_test, 1, config.sg_window_length, config.sg_polyorder)
X_sg2_train = savitzky_golay_derivative(X_raw_train, 2, config.sg_window_length, config.sg_polyorder)
X_sg2_test = savitzky_golay_derivative(X_raw_test, 2, config.sg_window_length, config.sg_polyorder)
X_snv_sg1_train = savitzky_golay_derivative(X_snv_train, 1, config.sg_window_length, config.sg_polyorder)
X_snv_sg1_test = savitzky_golay_derivative(X_snv_test, 1, config.sg_window_length, config.sg_polyorder)
X_snv_sg2_train = savitzky_golay_derivative(X_snv_train, 2, config.sg_window_length, config.sg_polyorder)
X_snv_sg2_test = savitzky_golay_derivative(X_snv_test, 2, config.sg_window_length, config.sg_polyorder)

train_feat, feature_catalog = build_feature_table(train_meta, X_raw_train, wavelengths, X_snv_train, X_sg1_train, X_sg2_train, X_snv_sg1_train, X_snv_sg2_train)
test_feat, _ = build_feature_table(test_meta, X_raw_test, wavelengths, X_snv_test, X_sg1_test, X_sg2_test, X_snv_sg1_test, X_snv_sg2_test)

train_feat = attach_probability_features(train_feat, find_probability_frame("train"))
test_feat = attach_probability_features(test_feat, find_probability_frame("test"))
for df in [train_feat, test_feat]:
    df["softwood_prob_x_sg1_2100"] = df["softwood_prob"] * df["sg1_2100"]
    df["entropy_x_sg1_2100"] = df["entropy_woodtype"] * df["sg1_2100"]

cnn_rows = [
    ("softwood_prob", "CNN", "CNN", "latent", "wood_type確率"),
    ("hardwood_prob", "CNN", "CNN", "latent", "wood_type確率"),
    ("entropy_woodtype", "CNN", "CNN", "latent", "wood_type不確実性"),
    ("uncertainty_woodtype", "CNN", "CNN", "latent", "1-max probability"),
    ("softwood_prob_x_sg1_2100", "CNN", "CNN_x_raw_sg1", "latent x 2100", "wood_type補正"),
    ("entropy_x_sg1_2100", "CNN", "CNN_x_raw_sg1", "latent x 2100", "境界サンプル補正"),
]
feature_catalog = pd.concat([feature_catalog, pd.DataFrame(cnn_rows, columns=["feature", "system", "preprocess", "wavelength", "meaning"])], ignore_index=True)
CANDIDATE_FEATURES = [f for f in feature_catalog["feature"] if f in train_feat.columns]

train_feat = train_feat.replace([np.inf, -np.inf], np.nan)
test_feat = test_feat.replace([np.inf, -np.inf], np.nan)
train_feat.to_csv(RESULT_DIR / "T12E_features_train.csv", index=False, encoding="utf-8-sig")
test_feat.to_csv(RESULT_DIR / "T12E_features_test.csv", index=False, encoding="utf-8-sig")
feature_catalog.to_csv(RESULT_DIR / "T12E_feature_catalog.csv", index=False, encoding="utf-8-sig")

print(train_feat.shape, test_feat.shape, "candidate_features:", len(CANDIDATE_FEATURES))
display(feature_catalog)
display(train_feat[[ID_COL, SPECIES_COL, TARGET_COL, *CANDIDATE_FEATURES[:8]]].head())

## GroupKFold SVR And Stepwise

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR


def make_svr_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale")),
    ])


def species_rmse_table(pred_df, pred_col="pred_svr"):
    return (
        pred_df.groupby(SPECIES_COL)
        .apply(lambda g: pd.Series({"n": len(g), "rmse": rmse(g[TARGET_COL], g[pred_col])}), include_groups=False)
        .reset_index()
        .sort_values("rmse", ascending=False)
    )


def evaluate_feature_set(feature_cols, label, return_oof=True):
    work = train_feat[[ID_COL, GROUP_COL, SPECIES_COL, TARGET_COL, *feature_cols]].replace([np.inf, -np.inf], np.nan).copy()
    work = work.loc[work[TARGET_COL].notna()].reset_index(drop=True)
    X = work[feature_cols]
    y = work[TARGET_COL].to_numpy(float)
    groups = work[SPECIES_COL]
    cv = GroupKFold(n_splits=max(2, min(N_SPLITS, int(groups.nunique()))))
    oof = np.full(len(work), np.nan)
    for tr_idx, va_idx in cv.split(X, y, groups):
        model = make_svr_pipeline()
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict(X.iloc[va_idx])
    pred_df = work[[ID_COL, GROUP_COL, SPECIES_COL, TARGET_COL]].copy()
    pred_df["pred_svr"] = oof
    species_df = species_rmse_table(pred_df)
    sp = species_df["rmse"].to_numpy(float)
    metrics = {
        "label": label,
        "features": "+".join(feature_cols),
        "n_features": len(feature_cols),
        "oof_rmse": rmse(y, oof),
        "species_mean_rmse": float(np.mean(sp)),
        "species_std_rmse": float(np.std(sp)),
        "species_min_rmse": float(np.min(sp)),
        "species_max_rmse": float(np.max(sp)),
        "species_range_rmse": float(np.max(sp) - np.min(sp)),
    }
    metrics["robust_score"] = metrics["species_mean_rmse"] + metrics["species_std_rmse"] + 0.25 * metrics["species_range_rmse"]
    return metrics, species_df, pred_df if return_oof else pd.DataFrame()


def prefilter_features(features):
    rows = []
    for feat in features:
        m, _, _ = evaluate_feature_set([feat], f"single__{feat}", return_oof=False)
        m["feature"] = feat
        rows.append(m)
    single_df = pd.DataFrame(rows).sort_values(["robust_score", "oof_rmse"]).reset_index(drop=True)
    selected = single_df["feature"].tolist() if TOP_K_PREFILTER is None else single_df.head(TOP_K_PREFILTER)["feature"].tolist()
    for must in ["sg1_2100", "softwood_prob", "entropy_woodtype", "uncertainty_woodtype"]:
        if must in features and must not in selected:
            selected.append(must)
    return selected, single_df


def run_forward_stepwise(features, objective="robust_score"):
    cache = {}
    selected, remaining = [], list(features)
    trials, trace = [], []
    best_obj = np.inf
    while remaining and len(selected) < MAX_STEPS:
        candidates = []
        for feat in remaining:
            trial = selected + [feat]
            key = tuple(trial)
            if key not in cache:
                cache[key] = evaluate_feature_set(trial, f"step{len(selected)+1}__add_{feat}")
            m = dict(cache[key][0])
            m["label"] = f"step{len(selected)+1}__add_{feat}"
            m["step"] = len(selected) + 1
            m["added_feature"] = feat
            m["selected_features"] = "+".join(trial)
            candidates.append(m)
            trials.append(m)
        step_df = pd.DataFrame(candidates).sort_values([objective, "oof_rmse", "species_mean_rmse"]).reset_index(drop=True)
        best = step_df.iloc[0].to_dict()
        improvement = best_obj - float(best[objective])
        if selected and improvement < MIN_IMPROVEMENT:
            print(f"stop: improvement {improvement:.6f} < {MIN_IMPROVEMENT}")
            break
        selected.append(best["added_feature"])
        remaining.remove(best["added_feature"])
        trace.append(best)
        best_obj = float(best[objective])
        print(f"step {len(selected):02d}: add {best['added_feature']} | robust={best_obj:.4f} | oof={best['oof_rmse']:.4f}")
    return pd.DataFrame(trials).sort_values([objective, "oof_rmse"]).reset_index(drop=True), pd.DataFrame(trace), cache


STEPWISE_FEATURES, single_feature_df = prefilter_features(CANDIDATE_FEATURES)
single_feature_df = single_feature_df.merge(feature_catalog, on="feature", how="left")
single_feature_df.to_csv(RESULT_DIR / "T12E_single_feature_prefilter.csv", index=False, encoding="utf-8-sig")
print("stepwise candidate count:", len(STEPWISE_FEATURES))
display(single_feature_df.head(30)[["feature", "system", "preprocess", "wavelength", "oof_rmse", "species_mean_rmse", "species_std_rmse", "robust_score"]])

baseline_metrics, baseline_species_rmse, baseline_oof = evaluate_feature_set(CANDIDATE_FEATURES, "baseline_all_candidates")
stepwise_trials, stepwise_trace, stepwise_cache = run_forward_stepwise(STEPWISE_FEATURES, objective="robust_score")
stepwise_summary = pd.concat([pd.DataFrame([baseline_metrics]), stepwise_trials], ignore_index=True, sort=False)
stepwise_summary = stepwise_summary.sort_values(["robust_score", "oof_rmse", "species_mean_rmse"]).reset_index(drop=True)

best_row = stepwise_summary.iloc[0]
best_features = str(best_row["features"]).split("+")
best_metrics, best_species_rmse, best_oof = evaluate_feature_set(best_features, "best_robust_stepwise")
selected_catalog = feature_catalog[feature_catalog["feature"].isin(best_features)].copy()
selected_catalog["selection_order"] = selected_catalog["feature"].map({f: i + 1 for i, f in enumerate(best_features)})
selected_catalog = selected_catalog.sort_values("selection_order")

single_feature_df.to_csv(RESULT_DIR / "T12E_single_feature_prefilter.csv", index=False, encoding="utf-8-sig")
stepwise_trials.to_csv(RESULT_DIR / "T12E_stepwise_all_trials.csv", index=False, encoding="utf-8-sig")
stepwise_trace.to_csv(RESULT_DIR / "T12E_stepwise_forward_trace.csv", index=False, encoding="utf-8-sig")
stepwise_summary.to_csv(RESULT_DIR / "T12E_stepwise_robustness_summary.csv", index=False, encoding="utf-8-sig")
best_species_rmse.to_csv(RESULT_DIR / "T12E_best_robust_species_rmse.csv", index=False, encoding="utf-8-sig")
best_oof.to_csv(RESULT_DIR / "T12E_best_robust_oof_predictions.csv", index=False, encoding="utf-8-sig")
selected_catalog.to_csv(RESULT_DIR / "T12E_best_robust_selected_feature_catalog.csv", index=False, encoding="utf-8-sig")

display(stepwise_trace[["step", "added_feature", "n_features", "oof_rmse", "species_mean_rmse", "species_std_rmse", "species_range_rmse", "robust_score"]])
display(stepwise_summary[["label", "n_features", "oof_rmse", "species_mean_rmse", "species_std_rmse", "species_range_rmse", "robust_score", "features"]].head(15))
display(selected_catalog)

## Visualize Robustness And Drying Curves

In [ ]:
if not stepwise_trace.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    sns.lineplot(data=stepwise_trace, x="n_features", y="oof_rmse", marker="o", ax=axes[0])
    sns.lineplot(data=stepwise_trace, x="n_features", y="species_mean_rmse", marker="o", label="mean", ax=axes[1])
    sns.lineplot(data=stepwise_trace, x="n_features", y="species_std_rmse", marker="o", label="std", ax=axes[1])
    sns.lineplot(data=stepwise_trace, x="n_features", y="robust_score", marker="o", ax=axes[2])
    axes[0].set_title("OOF RMSE")
    axes[1].set_title("Species RMSE mean/std")
    axes[2].set_title("Robust score")
    fig.tight_layout()
    savefig("T12E_stepwise_metric_trace.png")
    plt.show()
    plt.close("all")

plt.figure(figsize=(7, 5.5))
sns.scatterplot(data=stepwise_summary, x="species_mean_rmse", y="species_std_rmse", size="oof_rmse", hue="n_features", palette="viridis", sizes=(40, 220), alpha=0.75)
plt.scatter([baseline_metrics["species_mean_rmse"]], [baseline_metrics["species_std_rmse"]], marker="X", s=180, color="red", label="all candidates")
plt.scatter([best_metrics["species_mean_rmse"]], [best_metrics["species_std_rmse"]], marker="*", s=220, color="black", label="best robust")
plt.title("Robustness trade-off: species mean RMSE vs std")
plt.xlabel("species mean RMSE")
plt.ylabel("species RMSE std")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
savefig("T12E_stepwise_species_mean_vs_std.png")
plt.show()
plt.close("all")

plt.figure(figsize=(8, max(3, 0.35 * len(selected_catalog))))
sns.scatterplot(data=selected_catalog, x="selection_order", y="feature", hue="system", style="preprocess", s=90)
plt.title("Selected robust model features by order/system/preprocess")
plt.xlabel("selection order")
plt.ylabel("feature")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
savefig("T12E_selected_features_system_preprocess.png")
plt.show()
plt.close("all")

heat_rows = []
for label, feats in [("baseline_all_candidates", CANDIDATE_FEATURES), ("best_robust", best_features)]:
    _, sp, _ = evaluate_feature_set(feats, label, return_oof=False)
    tmp = sp[[SPECIES_COL, "rmse"]].copy()
    tmp["model"] = label
    heat_rows.append(tmp)
species_heat_df = pd.concat(heat_rows, ignore_index=True)
species_heat_df.to_csv(RESULT_DIR / "T12E_species_rmse_heatmap_table.csv", index=False, encoding="utf-8-sig")
heat = species_heat_df.pivot(index="model", columns=SPECIES_COL, values="rmse")
plt.figure(figsize=(13, max(3, 0.45 * len(heat))))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="mako_r", cbar_kws={"label": "RMSE"})
plt.title("Species RMSE heatmap")
savefig("T12E_species_rmse_heatmap.png")
plt.show()
plt.close("all")

compare_df = baseline_oof[[ID_COL, SPECIES_COL, TARGET_COL, "pred_svr"]].rename(columns={"pred_svr": "pred_all_candidates"})
compare_df = compare_df.merge(best_oof[[ID_COL, "pred_svr"]].rename(columns={"pred_svr": "pred_best_robust"}), on=ID_COL, how="left")
compare_df = compare_df.sort_values([SPECIES_COL, ID_COL]).copy()
compare_df["species_index"] = compare_df.groupby(SPECIES_COL, dropna=False).cumcount()
compare_df.to_csv(RESULT_DIR / "T12E_drying_curve_oof_baseline_vs_best.csv", index=False, encoding="utf-8-sig")

species_values = compare_df[SPECIES_COL].dropna().astype(str).unique().tolist()
ncols = 3 if len(species_values) > 6 else 2
nrows = math.ceil(len(species_values) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.9 * ncols, 2.35 * nrows), sharey=True)
axes = np.asarray(axes).reshape(-1)
for ax, species in zip(axes, species_values):
    g = compare_df[compare_df[SPECIES_COL].astype(str) == species]
    ax.scatter(g["species_index"], g[TARGET_COL], s=13, alpha=0.65, color="black", label="actual")
    ax.plot(g["species_index"], g["pred_all_candidates"], lw=1.1, color="tab:blue", label="all")
    ax.plot(g["species_index"], g["pred_best_robust"], lw=1.1, color="tab:orange", label="best")
    ax.set_title(f"{species}  n={len(g)}")
    ax.set_xlabel("index within species")
    ax.set_ylabel("moisture")
    ax.grid(True, alpha=0.25)
for ax in axes[len(species_values):]:
    ax.axis("off")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.suptitle("Drying curve comparison: all candidates vs best robust stepwise", y=0.995, fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.97))
savefig("T12E_drying_curve_baseline_vs_best_robust_grid.png")
plt.show()
plt.close("all")

## Train All, Test Prediction, Submission

In [ ]:
FINAL_FEATURES = best_features
final_model = make_svr_pipeline()
X_train = train_feat[FINAL_FEATURES].replace([np.inf, -np.inf], np.nan)
y_train = train_feat[TARGET_COL].to_numpy(float)
X_test = test_feat[FINAL_FEATURES].replace([np.inf, -np.inf], np.nan)
final_model.fit(X_train, y_train)
test_pred = final_model.predict(X_test)

submission = pd.DataFrame({ID_COL: test_feat[ID_COL].values, "pred": test_pred})
submission_path = RESULT_DIR / "submission_T12E_stepwise_best_robust.csv"
submission_latest_path = RESULT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False, encoding="utf-8-sig")
submission.to_csv(submission_latest_path, index=False, encoding="utf-8-sig")

test_curve_df = test_feat[[ID_COL, GROUP_COL, SPECIES_COL, *FINAL_FEATURES]].copy()
test_curve_df["pred"] = test_pred
test_curve_df = test_curve_df.sort_values([SPECIES_COL, ID_COL]).reset_index(drop=True)
test_curve_df["species_index"] = test_curve_df.groupby(SPECIES_COL, dropna=False).cumcount()
test_curve_df.to_csv(RESULT_DIR / "T12E_test_prediction_curve.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(11, 5))
sns.lineplot(data=test_curve_df, x="species_index", y="pred", hue=SPECIES_COL, marker="o", ms=3, lw=1.2)
plt.title("T12E best robust stepwise SVR test prediction curve by species")
plt.xlabel("index within species")
plt.ylabel("predicted moisture")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
savefig("T12E_test_prediction_curve_overlay.png")
plt.show()
plt.close("all")

print("T12E summary")
print(f"- candidate features: {len(CANDIDATE_FEATURES)}")
print(f"- best robust OOF RMSE: {best_metrics['oof_rmse']:.4f}")
print(f"- species mean RMSE: {best_metrics['species_mean_rmse']:.4f}")
print(f"- species std RMSE: {best_metrics['species_std_rmse']:.4f}")
print(f"- species range RMSE: {best_metrics['species_range_rmse']:.4f}")
print("- selected features:", FINAL_FEATURES)
print("saved:", submission_path)
display(submission.head())
display(test_curve_df.groupby(SPECIES_COL)["pred"].agg(["count", "mean", "std", "min", "max"]).reset_index())